In [1]:
import torch

print(torch.__version__)

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.5.1+cu121
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import torch
import pandas as pd
import numpy as np

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PyTorch: 2.5.1+cu121
CUDA Available: True
Using device: cuda
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=10
)

model.to(device)

print("DistilBERT loaded successfully")

c:\Users\ASUS\Downloads\NLP_Group_35\bert_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6128.17it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

DistilBERT loaded successfully


In [4]:
df = pd.read_csv(
    "../data/processed/customer_support_en.csv"
)

df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text_length,clean_text
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,544,dear customer support teamnni am writing to re...
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,534,dear customer support teamnni hope this messag...
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,605,dear customer support teamnni hope this messag...
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,677,dear support teamnni hope this message reaches...
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,646,dear customer supportnni hope this message rea...


In [5]:
print(df.shape)

print(df.columns)

print(df["queue"].unique())

(16338, 18)
Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6',
       'tag_7', 'tag_8', 'text_length', 'clean_text'],
      dtype='str')
<ArrowStringArray>
[              'Technical Support',           'Returns and Exchanges',
            'Billing and Payments',             'Sales and Pre-Sales',
 'Service Outages and Maintenance',                 'Product Support',
                      'IT Support',                'Customer Service',
                 'Human Resources',                 'General Inquiry']
Length: 10, dtype: str


In [6]:
df = df.sample(
    3000,
    random_state=42
)

print(df.shape)

(3000, 18)


In [7]:
df["text"] = (
    df["subject"].fillna("")
    + " "
    + df["body"].fillna("")
)


df[["text","queue"]].head()

,text,queue
14848,Query on Data Analytics Tools for Investment O...,IT Support
10388,Problems with Connection Customers are facing ...,IT Support
12919,"Hello Customer Support, I am inquiring about ...",Billing and Payments
14890,Found Issues with Secure Data Access in Hospit...,Technical Support
14625,Could you offer assistance on securing medica...,IT Support


In [8]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["queue"]
)


num_labels = len(label_encoder.classes_)


print("Classes:", num_labels)

print(label_encoder.classes_)

Classes: 10
['Billing and Payments' 'Customer Service' 'General Inquiry'
 'Human Resources' 'IT Support' 'Product Support' 'Returns and Exchanges'
 'Sales and Pre-Sales' 'Service Outages and Maintenance'
 'Technical Support']


In [9]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


print(len(train_texts))
print(len(test_texts))

2400
600


In [10]:
model_name = "distilbert-base-uncased"


tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [11]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)


test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [12]:
import torch


class CustomerSupportDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels


    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key,val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item


    def __len__(self):
        return len(self.labels)

In [13]:
train_dataset = CustomerSupportDataset(
    train_encodings,
    train_labels
)


test_dataset = CustomerSupportDataset(
    test_encodings,
    test_labels
)


print(len(train_dataset))
print(len(test_dataset))

2400
600


In [14]:
AutoModelForSequenceClassification.from_pretrained(
"distilbert-base-uncased"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1984.79it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


model.to(device)


print("Model loaded")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1706.94it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [16]:
def compute_metrics(pred):
    
    labels = pred.label_ids

    predictions = np.argmax(
        pred.predictions,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy
    }

In [17]:
from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available()
)



In [18]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [19]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.863881,1.889432,0.315000
2,1.732563,1.679206,0.371667
3,1.444523,1.668722,0.368333


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]


TrainOutput(global_step=900, training_loss=1.7634278276231554, metrics={'train_runtime': 179.8239, 'train_samples_per_second': 40.039, 'train_steps_per_second': 5.005, 'total_flos': 238475335680000.0, 'train_loss': 1.7634278276231554, 'epoch': 3.0})

In [21]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Accuracy
1.444523,1.668722,3,0.368333


{'eval_loss': 1.6687216758728027, 'eval_accuracy': 0.36833333333333335}

In [22]:
predictions = trainer.predict(test_dataset)

pred_labels = np.argmax(
    predictions.predictions,
    axis=1
)


print(
    classification_report(
        test_labels,
        pred_labels,
        target_names=label_encoder.classes_
    )
)

                                 precision    recall  f1-score   support

           Billing and Payments       0.69      0.77      0.73        57
               Customer Service       0.24      0.29      0.26        91
                General Inquiry       0.00      0.00      0.00         6
                Human Resources       0.00      0.00      0.00        12
                     IT Support       0.20      0.02      0.03        66
                Product Support       0.23      0.28      0.26       116
          Returns and Exchanges       0.00      0.00      0.00        32
            Sales and Pre-Sales       0.00      0.00      0.00        23
Service Outages and Maintenance       0.64      0.70      0.67        30
              Technical Support       0.39      0.57      0.46       167

                       accuracy                           0.37       600
                      macro avg       0.24      0.26      0.24       600
                   weighted avg       0.31      0

c:\Users\ASUS\Downloads\NLP_Group_35\bert_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\Downloads\NLP_Group_35\bert_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\Downloads\NLP_Group_35\bert_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [23]:
model.save_pretrained(
    "../models/member3/distilbert_model"
)


tokenizer.save_pretrained(
    "../models/member3/distilbert_tokenizer"
)


import pickle

with open(
    "../models/member3/distilbert_label_encoder.pkl",
    "wb"
) as f:
    pickle.dump(
        label_encoder,
        f
    )


print("DistilBERT model saved successfully")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s]

DistilBERT model saved successfully


In [24]:
text = """
My internet connection is not working
and I cannot access my account
"""


inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)


inputs = {
    k:v.to(device)
    for k,v in inputs.items()
}


model.eval()


with torch.no_grad():

    outputs = model(**inputs)


prediction = torch.argmax(
    outputs.logits,
    dim=1
)


print(
    "Predicted category:",
    label_encoder.inverse_transform(
        [prediction.item()]
    )[0]
)

Predicted category: Billing and Payments


In [25]:
import json

with open(
    "../reports/member3_distilbert_results.json",
    "w"
) as f:
    json.dump(
        results,
        f,
        indent=4
    )

In [26]:
import os

print(os.listdir("../models/member3"))
print(os.listdir("../reports"))

['distilbert_label_encoder.pkl', 'distilbert_model', 'distilbert_tokenizer', 'label_encoder.pkl', 'tfidf_vectorizer.pkl', 'xgboost.pkl']
['.gitkeep', 'eda_summary.md', 'member3_distilbert_results.json', 'member3_xgboost_results.md']


In [27]:
import json

summary = {
    "model": "DistilBERT",
    "base_model": "distilbert-base-uncased",
    "epochs": 3,
    "accuracy": results["eval_accuracy"],
    "eval_loss": results["eval_loss"],
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0)
}


with open(
    "../reports/member3_distilbert_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=4
    )


summary

{'model': 'DistilBERT',
 'base_model': 'distilbert-base-uncased',
 'epochs': 3,
 'accuracy': 0.36833333333333335,
 'eval_loss': 1.6687216758728027,
 'device': 'cuda',
 'gpu': 'NVIDIA GeForce RTX 3050 Laptop GPU'}

In [2]:
import torch
import pickle
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "../models/member3/distilbert_model"

tokenizer = AutoTokenizer.from_pretrained(
    "../models/member3/distilbert_tokenizer"
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path
)

with open(
    "../models/member3/distilbert_label_encoder.pkl",
    "rb"
) as f:
    label_encoder = pickle.load(f)

model.to(device)

print("Using device:", device)
print("Reload successful")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7448.65it/s]


Using device: cuda
Reload successful


In [3]:
text = """
I cannot login to my account.
The system keeps showing an error.
"""


inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)


inputs = {
    k:v.to(device)
    for k,v in inputs.items()
}


with torch.no_grad():

    output = model(**inputs)


pred = torch.argmax(
    output.logits,
    dim=1
)


print(
    label_encoder.inverse_transform(
        [pred.item()]
    )[0]
)

Product Support
